# Model 5 — Per-Chip Waveform Tuning
*CRISP-DM Phase 1 — Business Understanding*

## What this model does

The previous models (1 and 3) predict print quality averaged across all 30 chips.
This model goes one level deeper: it predicts `sd_std` for **each individual chip**.

This answers the question:
> "For chip 7 specifically, which waveform setting produces the most uniform nozzle output?"

That allows Canon to fine-tune settings per chip — useful when certain chips consistently
behave differently from the others.

## Key difference from Model 1

| | Model 1 | Model 5 |
|---|---|---|
| HeadIdx# used | No | Yes — as a feature |
| Predicts for | Any chip | A specific known chip |
| Generalises to new hardware | Yes | No |
| Use case | New printheads | Fine-tuning this printhead |

## 1. Load data
*CRISP-DM Phase 3 — Data Preparation*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_parquet('../../data/processed/waveform_tuning_row_summary.parquet')

condition_cols = ['Color$', 'HeadIdx#', 'V', 'F_r', 'dt2', 'Coverage#']
agg = df.groupby(condition_cols)[['sd_std', 'sd_mean']].mean().reset_index()

print(f'Rows: {len(agg):,}')
print(f'Chips: {agg["HeadIdx#"].nunique()}')
print(f'Unique waveform conditions (V, F_r, dt2): {agg[["V","F_r","dt2"]].drop_duplicates().shape[0]}')
agg.head(3)

## 2. Feature engineering
*CRISP-DM Phase 3 — Data Preparation*

Same interaction terms as Model 1, plus `HeadIdx#` one-hot encoded.
One-hot encoding turns chip number into 30 binary columns (chip_1, chip_2, ..., chip_30)
so the model can learn a separate offset for each chip.

In [ ]:
agg['V_x_Fr']          = agg['V'] * agg['F_r']
agg['dt2_x_coverage']  = agg['dt2'] * agg['Coverage#']
agg['V_sq']            = agg['V'] ** 2
agg['Fr_sq']           = agg['F_r'] ** 2

chip_dummies  = pd.get_dummies(agg['HeadIdx#'].astype(int), prefix='chip')
color_dummies = pd.get_dummies(agg['Color$'], prefix='color')

feature_cols = ['V', 'F_r', 'dt2', 'Coverage#',
                'V_x_Fr', 'dt2_x_coverage', 'V_sq', 'Fr_sq']

X = pd.concat([agg[feature_cols], chip_dummies, color_dummies], axis=1)
y = agg['sd_std']

print(f'Features: {X.shape[1]}  ({len(feature_cols)} waveform + {chip_dummies.shape[1]} chip + {color_dummies.shape[1]} color)')

## 3. Train / test split
*CRISP-DM Phase 3 — Data Preparation*

We split on **waveform conditions**, not on chips.

The goal is to predict the best setting for a chip we already know — so all chips
appear in both train and test. What we hold out is 20% of the (V, F_r, dt2) combinations,
to check whether the model generalises to untested settings on known chips.

In [ ]:
from sklearn.model_selection import train_test_split

# Get unique waveform conditions and split them
conditions = agg[['V', 'F_r', 'dt2']].drop_duplicates()
train_cond, test_cond = train_test_split(conditions, test_size=0.2, random_state=42)

train_mask = agg.set_index(['V','F_r','dt2']).index.isin(
    train_cond.set_index(['V','F_r','dt2']).index)

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f'Train: {len(X_train):,} rows  ({train_cond.shape[0]} waveform conditions × all chips)')
print(f'Test:  {len(X_test):,} rows  ({test_cond.shape[0]} waveform conditions × all chips)')

## 4. Train model
*CRISP-DM Phase 4 — Modeling*

Random Forest with 200 trees. No comparison needed — Random Forest already
outperformed all alternatives in Models 1–4.

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'R²:  {r2:.4f}')
print(f'MAE: {mae:.6f}')

## 4a. Predicted vs Actual — how close are the predictions?
*CRISP-DM Phase 5 — Evaluation*

Each dot is one test sample. A perfect model would have all dots on the diagonal line.
Points above the line = model overestimates; below = underestimates.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: predicted vs actual scatter ---
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.15, s=8, color='#00b4d8')
lim = [min(y_test.min(), y_pred.min()) - 0.0005,
       max(y_test.max(), y_pred.max()) + 0.0005]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual sd_std')
ax.set_ylabel('Predicted sd_std')
ax.set_title(f'Predicted vs Actual  (test set, R²={r2:.3f})')
ax.legend()

# --- Right: residual distribution ---
ax2 = axes[1]
residuals = y_pred - y_test
ax2.hist(residuals, bins=60, color='#00b4d8', edgecolor='none', alpha=0.8)
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
ax2.axvline(residuals.mean(), color='orange', linestyle='-', linewidth=1.5,
            label=f'Mean error = {residuals.mean():.5f}')
ax2.set_xlabel('Prediction error  (predicted − actual)')
ax2.set_ylabel('Count')
ax2.set_title('Distribution of prediction errors')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Errors are symmetric around zero: mean={residuals.mean():.6f}, std={residuals.std():.6f}')

## 4a-ii. Predicted vs Actual — per color
*CRISP-DM Phase 5 — Evaluation*

Same scatter plot split by ink color. This shows whether the model performs
consistently across all four colors or if it struggles more with one.

In [ ]:
test_df_color = agg[~train_mask].copy()
test_df_color['pred'] = y_pred

colors_order = ['M', 'C', 'Y', 'K']
color_names  = {'M': 'Magenta', 'C': 'Cyan', 'Y': 'Yellow', 'K': 'Black'}
dot_colors   = {'M': '#e91e8c', 'C': '#00b4d8', 'Y': '#f4c430', 'K': '#555555'}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for ax, color in zip(axes.flatten(), colors_order):
    sub = test_df_color[test_df_color['Color$'] == color]
    r2_c = r2_score(sub['sd_std'], sub['pred'])

    ax.scatter(sub['sd_std'], sub['pred'], alpha=0.2, s=8, color=dot_colors[color])
    lim = [min(sub['sd_std'].min(), sub['pred'].min()) - 0.0005,
           max(sub['sd_std'].max(), sub['pred'].max()) + 0.0005]
    ax.plot(lim, lim, 'r--', linewidth=1.5, label='Perfect prediction')
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual sd_std')
    ax.set_ylabel('Predicted sd_std')
    ax.set_title(f'{color_names[color]} ({color})  —  R²={r2_c:.3f}')
    ax.legend(fontsize=9)

plt.suptitle('Predicted vs Actual by color', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('R² per color:')
for color in colors_order:
    sub = test_df_color[test_df_color['Color$'] == color]
    print(f'  {color_names[color]:8s}: R²={r2_score(sub["sd_std"], sub["pred"]):.4f}  '
          f'  MAE={mean_absolute_error(sub["sd_std"], sub["pred"]):.6f}')

## 4b. Prediction error per chip — which chips are harder to predict?
*CRISP-DM Phase 5 — Evaluation*

Box plot of absolute prediction error grouped by chip.
A chip with a wide box or high median means the model is less certain about that chip.

In [ ]:
test_df = agg[~train_mask].copy()
test_df['pred']      = y_pred
test_df['abs_error'] = (test_df['pred'] - test_df['sd_std']).abs()

chip_errors   = [test_df[test_df['HeadIdx#'] == c]['abs_error'].values for c in range(1, 31)]
median_errors = [test_df[test_df['HeadIdx#'] == c]['abs_error'].median() for c in range(1, 31)]
worst_chips   = sorted(range(1, 31), key=lambda c: median_errors[c-1], reverse=True)[:5]

fig, ax = plt.subplots(figsize=(14, 5))
bp = ax.boxplot(chip_errors, patch_artist=True, medianprops=dict(color='red', linewidth=1.5))

for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#e74c3c' if (i+1) in worst_chips else '#00b4d8')
    patch.set_alpha(0.7)

ax.set_xticks(range(1, 31))
ax.set_xticklabels(range(1, 31))
ax.set_xlabel('Chip (HeadIdx#)')
ax.set_ylabel('Absolute prediction error')
ax.set_title('Prediction error per chip  (red = 5 hardest to predict)')
ax.axhline(test_df['abs_error'].median(), color='grey', linestyle='--', label='Overall median')
ax.legend()
plt.tight_layout()
plt.show()

## 4c. Feature importance — what drives the predictions?
*CRISP-DM Phase 5 — Evaluation*

Features are grouped into three categories:
- **Waveform** — the parameters Canon can tune (V, F_r, dt2 and their interactions)
- **Chip identity** — which of the 30 chips this row belongs to
- **Color** — ink color

The grouped bar shows the total importance of each category.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

waveform_imp = importances[[c for c in importances.index if c in feature_cols or c == 'Coverage#']].sum()
chip_imp     = importances[[c for c in importances.index if c.startswith('chip_')]].sum()
color_imp    = importances[[c for c in importances.index if c.startswith('color_')]].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: grouped importance ---
ax = axes[0]
groups  = ['Waveform\n(V, F_r, dt2…)', 'Chip identity\n(which chip)', 'Color\n(M/C/Y/K)']
vals    = [waveform_imp, chip_imp, color_imp]
colors_ = ['#00b4d8', '#ffd666', '#6ae59b']
bars = ax.bar(groups, vals, color=colors_, width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Total feature importance')
ax.set_title('Importance by feature group')
ax.set_ylim(0, max(vals) + 0.08)

# --- Right: top 15 individual features ---
ax2 = axes[1]
top15 = importances.head(15)
bar_colors = ['#ffd666' if c.startswith('chip_') else
              '#6ae59b' if c.startswith('color_') else '#00b4d8'
              for c in top15.index]
ax2.barh(top15.index[::-1], top15.values[::-1], color=bar_colors[::-1])
ax2.set_xlabel('Feature importance')
ax2.set_title('Top 15 individual features')

from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color='#00b4d8', label='Waveform'),
                    Patch(color='#ffd666', label='Chip identity'),
                    Patch(color='#6ae59b', label='Color')], loc='lower right')

plt.tight_layout()
plt.show()

print(f'Waveform parameters account for {waveform_imp:.1%} of predictive power')
print(f'Chip identity accounts for       {chip_imp:.1%}')
print(f'Color accounts for               {color_imp:.1%}')

## 4d. Why per-chip tuning matters — same setting, different chips
*CRISP-DM Phase 5 — Evaluation*

For one fixed waveform setting, this plot shows how `sd_std` varies across all 30 chips.
If all chips behaved identically, you would see a flat horizontal line.
The spread you see here is exactly why a one-size-fits-all setting is suboptimal.

In [ ]:
# Pick the most common (Color, Coverage, V, F_r, dt2) combination
common = (agg.groupby(['Color$', 'Coverage#', 'V', 'F_r', 'dt2'])
             .size().idxmax())
color_fix, cov_fix, v_fix, fr_fix, dt2_fix = common

subset = agg[
    (agg['Color$']    == color_fix) &
    (agg['Coverage#'] == cov_fix)   &
    (agg['V']         == v_fix)     &
    (agg['F_r']       == fr_fix)    &
    (agg['dt2']       == dt2_fix)
].sort_values('HeadIdx#')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: actual sd_std per chip for one fixed setting ---
ax = axes[0]
mean_val = subset['sd_std'].mean()
chip_colors = ['#e74c3c' if v > mean_val * 1.2 else '#00b4d8' for v in subset['sd_std']]
ax.bar(subset['HeadIdx#'].astype(str), subset['sd_std'], color=chip_colors)
ax.axhline(mean_val, color='grey', linestyle='--', label=f'Mean = {mean_val:.4f}')
ax.set_xlabel('Chip (HeadIdx#)')
ax.set_ylabel('sd_std')
ax.set_title(f'sd_std per chip for one fixed setting\n(Color={color_fix}, Coverage={cov_fix}, V={v_fix}, F_r={fr_fix}, dt2={dt2_fix})')
ax.legend()

# --- Right: range of sd_std across all settings for each chip ---
chip_ranges = agg.groupby('HeadIdx#')['sd_std'].agg(['min', 'max', 'mean']).reset_index()
ax2 = axes[1]
ax2.vlines(chip_ranges['HeadIdx#'], chip_ranges['min'], chip_ranges['max'],
           color='#00b4d8', linewidth=2, alpha=0.7, label='min–max range')
ax2.scatter(chip_ranges['HeadIdx#'], chip_ranges['mean'],
            color='#ffd666', zorder=5, s=40, label='mean')
ax2.set_xlabel('Chip (HeadIdx#)')
ax2.set_ylabel('sd_std')
ax2.set_title('Range of sd_std per chip across all settings\n(wide range = waveform has strong effect on this chip)')
ax2.legend()

plt.tight_layout()
plt.show()

spread = subset['sd_std'].max() - subset['sd_std'].min()
print(f'Fixed setting: same waveform → sd_std ranges from {subset["sd_std"].min():.4f} to {subset["sd_std"].max():.4f} across chips')
print(f'Spread: {spread:.4f}  — this is what per-chip tuning can reduce')

## 5. Results — per-chip best settings
*CRISP-DM Phase 5 — Evaluation*

For each chip and each (Color, Coverage) combination, find the waveform setting
that minimises predicted `sd_std`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# Predict sd_std for all known conditions
X_all = pd.concat([agg[feature_cols], chip_dummies, color_dummies], axis=1)
agg['sd_std_pred'] = model.predict(X_all)

# For each chip × Color × Coverage, find the best setting
best = (agg.sort_values('sd_std_pred')
          .drop_duplicates(subset=['HeadIdx#', 'Color$', 'Coverage#'])
          [['HeadIdx#', 'Color$', 'Coverage#', 'V', 'F_r', 'dt2', 'sd_std_pred', 'sd_std']]
          .rename(columns={'sd_std_pred': 'predicted_std', 'sd_std': 'actual_std'})
          .sort_values(['Color$', 'Coverage#', 'HeadIdx#'])
          .reset_index(drop=True))

print('Best setting per chip × Color × Coverage (first 20 rows):')
best.head(20)

## 6. Which chips are hardest to control?
*CRISP-DM Phase 5 — Evaluation*

Compare the best achievable `sd_std` per chip — chips with a higher minimum
are inherently harder to make uniform regardless of waveform settings.

In [ ]:
chip_summary = (best.groupby('HeadIdx#')['predicted_std']
                    .mean()
                    .reset_index()
                    .rename(columns={'predicted_std': 'avg_best_std'})
                    .sort_values('avg_best_std'))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['#e74c3c' if v > chip_summary['avg_best_std'].quantile(0.75) else '#2ecc71'
          for v in chip_summary['avg_best_std']]
ax.bar(chip_summary['HeadIdx#'].astype(str), chip_summary['avg_best_std'], color=colors)
ax.set_xlabel('Chip (HeadIdx#)')
ax.set_ylabel('Avg best sd_std')
ax.set_title('Best achievable sd_std per chip (averaged over Color × Coverage)\nRed = hardest to control')
ax.axhline(chip_summary['avg_best_std'].mean(), color='grey', linestyle='--', label='mean')
ax.legend()
plt.tight_layout()
plt.show()

print('Top 5 hardest chips:')
print(chip_summary.tail(5).to_string(index=False))

## 7. Find best setting for a specific chip and target SD
*CRISP-DM Phase 5 — Evaluation*

Given a chip number, color, coverage, and desired SD level — find the settings
that get closest to the target while minimising within-chip variability.

In [ ]:
target_sd   = 0.35
tolerance   = 0.02
chip        = 5
color       = 'C'
coverage    = 22.0

filtered = agg[
    (agg['HeadIdx#'] == chip) &
    (agg['Color$']   == color) &
    (agg['Coverage#'] == coverage) &
    (agg['sd_mean'] >= target_sd - tolerance) &
    (agg['sd_mean'] <= target_sd + tolerance)
]

if filtered.empty:
    print(f'No settings found for chip {chip}, {color}, coverage {coverage} near SD={target_sd}.')
    print('Try widening the tolerance or changing the target.')
else:
    result = (filtered.sort_values('sd_std_pred')
                      [['V', 'F_r', 'dt2', 'sd_mean', 'sd_std_pred']]
                      .rename(columns={'sd_mean': 'actual_sd_mean', 'sd_std_pred': 'predicted_std'})
                      .head(10)
                      .reset_index(drop=True))
    print(f'Best settings for chip {chip}, Color={color}, Coverage={coverage}, target SD≈{target_sd}:')
    print(result.to_string())